**⚠️ Technical Note:** This notebook was developed in a Google Colab environment. The raw bioclimatic and GEDI/VHM datasets are hosted in a private Google Drive directory. To replicate this study, users must provide their own raster assets or contact the author for access to the standardized 1km² Parquet files.

# 04: IPCC CMIP6 Data Acquisition & Aggregation
**Project:** A Validated Predictive Framework for Climate-Smart Reforestation in Armenia

**Author:** Narek Ohanyan

In [13]:
!pip install intake-esm gcsfs xarray dask netCDF4 zarr cftime

In [14]:
import intake
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# Suppress warnings for cleaner output
import warnings
warnings.filterwarnings('ignore')

In [15]:
# Open the Pangeo CMIP6 catalog
url = "https://storage.googleapis.com/cmip6/pangeo-cmip6.json"
col = intake.open_esm_datastore(url)

# Define our query parameters
query = dict(
    experiment_id=['historical', 'ssp126', 'ssp245', 'ssp370', 'ssp585'],
    table_id='day',
    variable_id=['tasmax', 'pr', 'hurs'],
    member_id='r1i1p1f1',
    source_id='MPI-ESM1-2-LR'
)

# Search the catalog
cat_subset = col.search(**query)
print(f"Datasets found: {len(cat_subset.df)}")

# Load data into an xarray dataset dictionary (lazy loading via dask)
# ADDED: storage_options={'token': 'anon'} to bypass Google credential checks
dset_dict = cat_subset.to_dataset_dict(
    zarr_kwargs={'consolidated': True},
    storage_options={'token': 'anon'}
)

Datasets found: 15

--> The keys in the returned dictionary of datasets are constructed as follows:
	'activity_id.institution_id.source_id.experiment_id.table_id.grid_label'


<div><progress max="5" value="5"></progress> 100.00% [5/5 00:07&lt;00:00]</div>

In [16]:
def subset_armenia(ds):
    """Subsets the dataset to the approximate bounding box of Armenia."""
    lat_bnds, lon_bnds = [38.8, 41.3], [43.4, 46.6]

    # Handle longitude conventions (0-360 vs -180 to 180)
    if ds.lon.max() > 180:
        ds = ds.assign_coords(lon=(ds.lon + 180) % 360 - 180)
        ds = ds.sortby(ds.lon)

    ds_sub = ds.sel(lat=slice(lat_bnds[0], lat_bnds[1]), lon=slice(lon_bnds[0], lon_bnds[1]))
    return ds_sub.mean(dim=['lat', 'lon']) # Return spatial average

def calculate_vpd(tasmax_k, hurs):
    """
    Calculates Vapor Pressure Deficit (VPD) in kPa.
    tasmax_k: Max temp in Kelvin
    hurs: Relative humidity in %
    """
    # Convert Temp to Celsius
    t_c = tasmax_k - 273.15

    # Saturation Vapor Pressure (es)
    es = 0.6108 * np.exp((17.27 * t_c) / (t_c + 237.3))

    # VPD calculation
    vpd = es * (1 - (hurs / 100))
    return vpd

In [19]:
periods = {
    'historical': slice('1995', '2014'),
    'mid_term': slice('2041', '2060'),
    'long_term': slice('2081', '2100')
}

scenarios = ['ssp126', 'ssp245', 'ssp370', 'ssp585']
results = {}

for key, ds in dset_dict.items():
    exp = ds.attrs['experiment_id']

    # 1. Spatial Subset to Armenia (using your spatial average function)
    ds_sub = subset_armenia(ds)

    # 2. Isolate Growing Season (May 1st to Sept 30th)
    ds_gs = ds_sub.where(ds_sub['time'].dt.month.isin([5, 6, 7, 8, 9]), drop=True)

    # 3. UNIT CONVERSION: Precipitation Flux -> Monthly Average (mm/month)
    # 1 mean month ≈ 2,629,800 seconds
    if 'pr' in ds_gs:
        ds_gs['pr_mm_monthly'] = ds_gs['pr'] * 2629800

    # 4. UNIT CONVERSION: VPD (kPa)
    if 'tasmax' in ds_gs and 'hurs' in ds_gs:
        ds_gs['vpd_kpa'] = calculate_vpd(ds_gs['tasmax'], ds_gs['hurs'])

    results[exp] = ds_gs

# --- CALCULATE DELTAS ---

# Compute the Reference Mean for the 1995-2014 Baseline
hist_ds = results['historical'].sel(time=periods['historical']).mean(dim='time').compute()

table_data = []
for ssp in scenarios:
    for p_key in ['mid_term', 'long_term']:
        # Select the specific 20-year future window
        future_ds = results[ssp].sel(time=periods[p_key]).mean(dim='time').compute()

        # Calculate consistent deltas
        dt = float(future_ds['tasmax'] - hist_ds['tasmax'])
        dp = float(future_ds['pr_mm_monthly'] - hist_ds['pr_mm_monthly']) # mm/month
        dv = float(future_ds['vpd_kpa'] - hist_ds['vpd_kpa'])

        table_data.append({
            'Time Horizon': '2041-2060' if p_key == 'mid_term' else '2081-2100',
            'Pathway': ssp.upper(),
            'ΔTmax [°C]': round(dt, 2),
            'ΔP [mm/mo]': round(dp, 2), # Corrected Unit
            'ΔVPD [kPa]': round(dv, 3)
        })

# Create and display the final projection matrix
projection_matrix = pd.DataFrame(table_data)
display(projection_matrix)

,Time Horizon,Pathway,ΔTmax [°C],ΔP [mm/mo],ΔVPD [kPa]
0,2041-2060,SSP126,1.02,-1.94,0.168
1,2081-2100,SSP126,1.08,-1.28,0.194
2,2041-2060,SSP245,1.79,-4.09,0.271
3,2081-2100,SSP245,2.41,-4.44,0.417
4,2041-2060,SSP370,2.71,-9.04,0.511
5,2081-2100,SSP370,5.44,-13.50,1.073
6,2041-2060,SSP585,1.97,-6.20,0.277
7,2081-2100,SSP585,5.72,-9.83,1.045
